# Milestone 6 — Baseline training on Google Colab

This notebook runs the **frozen-base MobileNetV2 baseline** (`src/train.py`)
on Google Colab and stores every result on **Google Drive** so nothing is lost
when the runtime ends.

What it does:

1. Inspects the runtime and installs a compatible TensorFlow **only if needed**
2. Clones this repository at a chosen revision (pinnable to a commit SHA)
3. Downloads + inspects RealWaste with the existing `src/` scripts
4. **Validates** the committed split manifests (never regenerates them)
5. Trains only the classification head (backbone frozen), with early stopping
6. Saves `best_model.keras` + honest run reports to Drive, ready for download

What it does **not** do (later milestones): fine-tuning, test-set evaluation,
the Streamlit app.

> **Honesty note:** this notebook has been validated *locally* (valid JSON,
> every code cell passes a Python syntax check, `tests/test_notebook.py`
> passes). It has **not** been executed in Google Colab, so it contains **no
> accuracy numbers**. Real numbers must come from an actual run and live in
> `run_metadata.json`.

**Runtime:** `Runtime → Change runtime type → T4 GPU` is recommended
(training still works on CPU, only slower).

## Step 1 — Inspect the Colab runtime before installing anything

Decide what to install **before** importing TensorFlow:

- The repo was developed on **Python 3.11 + TensorFlow 2.15.1 (Keras 2)** —
  the versions pinned in `requirements.lock.txt`.
- **Never install `requirements.lock.txt` in Colab.** It contains
  Windows-only pins (e.g. `win32-*` packages) that do not work on Linux.
- Policy: on a Python 3.11 runtime install the repo-tested
  `tensorflow==2.15.1`; on any other runtime install the newest TensorFlow
  that matches that Python, and record the difference honestly — the run's
  `run_metadata.json` stores the exact versions actually used.

This cell only reads package metadata; it does **not** import TensorFlow.

In [ ]:
# Step 1: inspect environment (version metadata only — no TensorFlow import).
import platform
from importlib import metadata


def pkg_version(name):
    try:
        return metadata.version(name)
    except metadata.PackageNotFoundError:
        return None


python_version = platform.python_version()
tf_version = pkg_version("tensorflow")
keras_version = pkg_version("keras")
numpy_version = pkg_version("numpy")

print("Python    :", python_version)
print("TensorFlow:", tf_version or "NOT installed")
print("Keras pkg :", keras_version or "not found as a package")
print("NumPy     :", numpy_version or "NOT installed")
print()
print("Repo-tested stack: Python 3.11 / TensorFlow 2.15.1 / Keras 2.15.0.")

## Step 2 — Install only if needed (restart afterwards if you install)

If Step 1 printed **"Will install: ..."**, run the two cells below:

1. the install cell actually installs TensorFlow,
2. the restart cell **restarts the runtime** — this is mandatory after any
   install, otherwise Python keeps using the old already-imported modules and
   you get a confusing half-old/half-new environment.

If Step 1 printed "nothing to install", you can skip both.

In [ ]:
# Step 2a: decide whether anything needs installing.
from pathlib import Path

INSTALL_MARKER = Path("/content/.tf_install_requested")


def parse_version(text):
    return tuple(int(part) for part in text.split(".")[:2])


needs_install = False
if tf_version is None:
    needs_install = True
elif parse_version(tf_version) < (2, 15):
    # Too old for this code base (it uses the tf.keras API of >= 2.15).
    needs_install = True

if not needs_install:
    INSTALL_MARKER.unlink(missing_ok=True)
    print(f"TensorFlow {tf_version} is already usable - nothing to install.")
    target = None
else:
    # Python 3.11 -> repo-tested pin for reproducibility; anything else ->
    # newest TF for that runtime (the difference is recorded in run metadata).
    target = (
        "tensorflow==2.15.1" if python_version.startswith("3.11") else "tensorflow"
    )
    INSTALL_MARKER.write_text(target, encoding="utf-8")
    print(f"Will install: {target}")
    print("Run the next cell, then the restart cell.")

In [ ]:
# Step 2b: perform the planned install (skips when the marker is absent).
import subprocess
import sys

if INSTALL_MARKER.exists():
    target = INSTALL_MARKER.read_text(encoding="utf-8")
    print(f"Installing {target} ... (takes a few minutes)")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", target], check=True
    )
    print("Install finished. NOW run the restart cell below.")
else:
    print("No install planned - skipping.")

In [ ]:
# Step 2c: restart the runtime ONLY if something was installed.
# After the restart, notebook variables are cleared: re-run Step 1 and Step 2a
# (they will now report "nothing to install"), then continue from Step 3.
from pathlib import Path

marker = Path("/content/.tf_install_requested")
if marker.exists():
    marker.unlink()
    print("Packages were installed - restarting the runtime now.")
    print("After the restart: re-run Step 1 + Step 2a, then continue at Step 3.")
    get_ipython().kernel.do_shutdown(restart=True)
else:
    print("No restart needed - continue with Step 3.")

## Step 3 — Verify the environment actually imports

Now TensorFlow is imported for the first time. The versions printed here are
the ones that will be recorded in `run_metadata.json` (`environment` section),
so a later reader can tell exactly what produced the model.

In [ ]:
# Step 3: import and verify.
import keras
import numpy as np
import tensorflow as tf

print("TensorFlow:", tf.__version__)
print("Keras     :", keras.__version__)
print("NumPy     :", np.__version__)

major, minor = (int(part) for part in tf.__version__.split(".")[:2])
assert (major, minor) >= (2, 15), (
    f"TensorFlow {tf.__version__} is too old for this code base (need >= 2.15)."
)

gpus = tf.config.list_physical_devices("GPU")
print("GPU(s)    :", gpus if gpus else "none detected (CPU only - training slower)")

## Step 4 — Mount Google Drive

Colab's `/content` is **wiped when the runtime ends**. Trained weights and run
reports must live on Drive (or be downloaded before the runtime dies) — losing
a run to a disconnect is the classic Colab mistake.

In [ ]:
# Step 4: mount Drive (safe to re-run).
from google.colab import drive

drive.mount("/content/drive")
print("Drive mounted.")

## Step 5 — Repository, revision, and output locations

- **`REVISION`**: for an exactly reproducible run, pin a full commit SHA
  (run `git rev-parse HEAD` in your local clone). The default `"main"` tracks
  the latest push — convenient, but a later push would change what this
  notebook runs, so record the resolved SHA printed in Step 6.
- Outputs go to `Drive/...-outputs/runs/`, deliberately **outside** the
  clone, so the checkout stays clean (`run_metadata.json` records
  `uncommitted_changes` and a dirty checkout would weaken reproducibility).

In [ ]:
# Step 5: configuration.
from pathlib import Path

REPO_URL = "https://github.com/snadeem03/ai-waste-classification-assistant.git"
REPO_NAME = "ai-waste-classification-assistant"
# Pin to a full commit SHA for a reproducible run, e.g. REVISION = "<40-hex>".
REVISION = "main"  # tracks the latest push - NOT pinned.

REPO_DIR = Path("/content") / REPO_NAME
BASE_DIR = Path("/content/drive/MyDrive") / f"{REPO_NAME}-outputs"
RUNS_DIR = BASE_DIR / "runs"  # best_model.keras + history + metadata + logs
BASE_DIR.mkdir(parents=True, exist_ok=True)
RUNS_DIR.mkdir(parents=True, exist_ok=True)

print("Repository dir :", REPO_DIR)
print("Checked out    :", REVISION)
print("Persistent dir :", BASE_DIR)
print("Runs dir       :", RUNS_DIR)

## Step 6 — Clone and check out the revision

Idempotent: re-running fetches updates instead of cloning twice.

In [ ]:
# Step 6: clone (or update) the repository and check out REVISION.
from pathlib import Path
import subprocess

# Normalize BEFORE any path operation. REPO_DIR must be a pathlib.Path
# here even if an earlier cell (or an edit) left it as a plain string -
# otherwise `REPO_DIR / ".git"` raises TypeError: unsupported operand
# type(s) for /: 'str' and 'str'.
REPO_DIR = Path(REPO_DIR)


def git(*args, cwd=None):
    return subprocess.run(
        ["git", *args],
        cwd=str(cwd or REPO_DIR),
        capture_output=True,
        text=True,
        check=True,
    )


if not (REPO_DIR / ".git").exists():
    print("Cloning ...")
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
else:
    print("Repository already present - fetching latest refs.")
    git("fetch", "origin")

git("checkout", REVISION)
if REVISION == "main":
    git("pull", "--ff-only", "origin", "main")
print("Checkout done.")

In [ ]:
# Step 6b: report exactly which commit this run uses.
import subprocess

resolved = subprocess.run(
    ["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"],
    capture_output=True,
    text=True,
    check=True,
).stdout.strip()
dirty = subprocess.run(
    ["git", "-C", str(REPO_DIR), "status", "--porcelain"],
    capture_output=True,
    text=True,
    check=True,
).stdout.strip()

print("HEAD            :", resolved)
print("Uncommitted work:", "yes - run is NOT exactly reproducible" if dirty else "no")

if REVISION != "main" and not resolved.startswith(REVISION):
    raise RuntimeError(f"Requested revision {REVISION} but checked out {resolved}.")
if REVISION == "main":
    print('Note: REVISION="main" is unpinned - record the SHA above with your results.')

## Step 7 — Download and inspect the dataset

Uses the existing scripts (`src/download_data.py`, `src/inspect_data.py`)
exactly as on a local machine — no notebook-specific download logic. The
download is idempotent: if `data/raw/` already holds the verified archive and
extraction, nothing is re-downloaded (~688 MB).

In [ ]:
# Step 7: download RealWaste (idempotent) and re-run the inspection report.
import json
import subprocess
import sys


def run_script(name, *args):
    cmd = [sys.executable, str(REPO_DIR / "src" / name), *args]
    print("Running:", " ".join(cmd))
    subprocess.run(cmd, cwd=str(REPO_DIR), check=True)


run_script("download_data.py")
run_script("inspect_data.py")

summary = json.loads(
    (REPO_DIR / "data" / "metadata" / "inspection_summary.json").read_text(
        encoding="utf-8"
    )
)
print()
print("selected total (actual) :", summary["actual_selected_total"])
print("selected total (expected):", summary["expected_selected_total"])
print("invalid images           :", summary["image_validation"]["invalid_image_count"])
print("class order              :", summary["class_order"])

## Step 8 — Validate the committed split manifests (never regenerate them)

`data/metadata/*.csv` are **reviewed artifacts**: their SHA-256 checksums and
row counts are recorded in `split_summary.json`. Regenerating them here would
silently redefine the split and break comparability with every earlier
milestone, so instead we verify:

1. checksums of `train.csv` / `validation.csv` / `test.csv` still match the
   recorded ones (byte-integrity of the committed files only — no images and
   no labels are used for any decision here),
2. `train` and `validation` load through `src/data_pipeline.py` with the
   expected row counts — this is exactly the loader training will use.

Training later opens **only** `train.csv` + `validation.csv`; `run_metadata.json`
records `data.test_manifest_used: false` to prove it.

In [ ]:
# Step 8: verify committed manifests against split_summary.json.
import hashlib
import json
import sys

sys.path.insert(0, str(REPO_DIR / "src"))
import data_pipeline


def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for chunk in iter(lambda: handle.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()


metadata_dir = REPO_DIR / "data" / "metadata"
summary = json.loads((metadata_dir / "split_summary.json").read_text(encoding="utf-8"))

for name in ("train.csv", "validation.csv", "test.csv"):
    actual = sha256_file(metadata_dir / name)
    expected = summary["manifest_checksums_sha256"][name]
    if actual != expected:
        raise RuntimeError(f"{name} checksum mismatch: {actual} != {expected}")
    print(f"{name}: checksum OK ({actual[:12]}...)")

splits = data_pipeline.load_splits(
    metadata_dir=metadata_dir,
    path_root=REPO_DIR,
    splits=("train", "validation"),
)
for split in ("train", "validation"):
    count = len(splits[split])
    expected = summary["split_totals"][split]
    if count != expected:
        raise RuntimeError(f"{split}: {count} rows != expected {expected}")
    print(f"{split}: {count} rows OK")

print("All committed manifests validated - none were regenerated.")

## Step 9 — Accelerator check

Record what hardware this run actually gets (also captured automatically in
`run_metadata.json` → `environment`).

In [ ]:
# Step 9: accelerator details.
import subprocess

try:
    result = subprocess.run(
        ["nvidia-smi"], capture_output=True, text=True, check=True, timeout=60
    )
    print(result.stdout)
except (FileNotFoundError, subprocess.CalledProcessError) as exc:
    print("nvidia-smi unavailable:", exc)

import tensorflow as tf

gpus = tf.config.list_physical_devices("GPU")
print("TensorFlow GPUs:", gpus if gpus else "none (CPU only)")

## Step 10 — Run the baseline training

This executes `src/train.py` as a subprocess:

- **Only the classification head trains**; the MobileNetV2 backbone stays
  frozen (ImageNet weights, downloaded on first use).
- Settings come from `configs/training.json`: seed 42, 224×224, batch 16,
  lr 0.001, max 15 epochs, dropout 0.2.
- `EarlyStopping(patience=3, restore_best_weights=True)` on `val_loss`;
  the best epoch's weights are saved to `best_model.keras`.
- Only `train.csv` + `validation.csv` are opened — **no test-set access,
  no fine-tuning** (later milestones).
- Everything is written to **Drive**: `RUNS_DIR/baseline_<timestamp>/`
  (`best_model.keras`, `history.csv`, `run_metadata.json`, `plots/`, logs).
- `PYTHONHASHSEED` is set for the child process, matching the local workflow.

Expected time: minutes on a T4 GPU; much longer on CPU.

In [ ]:
# Step 10: baseline training (frozen base, validation only, no test access).
import json
import os
import subprocess
import sys

config = json.loads(
    (REPO_DIR / "configs" / "training.json").read_text(encoding="utf-8")
)
SEED = config["seed"]

env = dict(os.environ, PYTHONHASHSEED=str(SEED))
cmd = [
    sys.executable,
    str(REPO_DIR / "src" / "train.py"),
    "--output-dir",
    str(RUNS_DIR),
]
print("Running:", " ".join(cmd))
result = subprocess.run(cmd, cwd=str(REPO_DIR), env=env)
if result.returncode != 0:
    raise RuntimeError("Training failed - see the log output above.")

run_dirs = [p for p in RUNS_DIR.glob("baseline_*") if p.is_dir()]
if not run_dirs:
    raise RuntimeError(f"No run directory found under {RUNS_DIR}")
RUN_DIR = max(run_dirs, key=lambda p: p.stat().st_mtime)
print("Run directory:", RUN_DIR)

## Step 11 — Show the curves and the best validation results

Everything below is **read from the run's files** (`history.csv`,
`run_metadata.json`) — numbers are never typed by hand. These are
**validation** metrics; the test split was not touched.

In [ ]:
# Step 11: plot curves + print the recorded best validation results.
import csv
import json

import matplotlib.pyplot as plt

metadata = json.loads((RUN_DIR / "run_metadata.json").read_text(encoding="utf-8"))
with open(RUN_DIR / "history.csv", newline="", encoding="utf-8") as handle:
    rows = list(csv.DictReader(handle))

epochs = [int(row["epoch"]) for row in rows]
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(epochs, [float(r["loss"]) for r in rows], label="train")
axes[0].plot(epochs, [float(r["val_loss"]) for r in rows], label="validation")
axes[0].set_title("Loss")
axes[0].set_xlabel("epoch")
axes[0].legend()
axes[0].grid(alpha=0.3)
axes[1].plot(epochs, [float(r["accuracy"]) for r in rows], label="train")
axes[1].plot(epochs, [float(r["val_accuracy"]) for r in rows], label="validation")
axes[1].set_title("Accuracy")
axes[1].set_xlabel("epoch")
axes[1].legend()
axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.show()

best = metadata["training"]["best"]
print("Run id            :", metadata["run_id"])
print("Epochs completed  :", metadata["configuration"]["epochs_completed"],
      "(max", str(metadata["configuration"]["epochs_requested"]) + ")")
print("Best epoch        :", best["epoch"])
print("Validation loss   :", round(best["val_loss"], 4))
print("Validation accuracy:", round(best["val_accuracy"], 4))
print("Class order       :", metadata["class_order"])
print("Git commit        :", metadata["git"]["commit"])
print("Uncommitted files :", metadata["git"]["uncommitted_changes"])
print("Model sha256      :", metadata["model"]["sha256"][:16] + "...")
print("Test set used     :", metadata["data"]["test_manifest_used"])

## Step 12 — Package the small reports + the model for transport

- `--export-reports` copies the *small* report files (`run_metadata.json`,
  `class_order.json`, `history.csv`, `environment_freeze.txt`, `plots/`) into
  the clone at `models/metadata/runs/<run_id>/` — **never** a model binary.
- Then we zip (a) those reports and (b) `best_model.keras` into Drive so both
  can be downloaded. The model zip stays out of Git (`.gitignore` blocks
  `*.keras` and `*.zip`); the report files may be committed to Git locally
  after download, which is exactly what this milestone expects.

In [ ]:
# Step 12: export small reports into the clone and zip both bundles to Drive.
import shutil
import subprocess
import sys
import zipfile

subprocess.run(
    [sys.executable, str(REPO_DIR / "src" / "train.py"),
     "--export-reports", str(RUN_DIR)],
    cwd=str(REPO_DIR),
    check=True,
)

reports_dir = REPO_DIR / "models" / "metadata" / "runs" / RUN_DIR.name
reports_zip = shutil.make_archive(
    str(BASE_DIR / f"reports_{RUN_DIR.name}"), "zip", reports_dir
)

model_zip = str(BASE_DIR / f"model_{RUN_DIR.name}.zip")
with zipfile.ZipFile(model_zip, "w", zipfile.ZIP_DEFLATED) as archive:
    archive.write(RUN_DIR / "best_model.keras", arcname="best_model.keras")
    archive.write(RUN_DIR / "class_order.json", arcname="class_order.json")
    archive.write(RUN_DIR / "run_metadata.json", arcname="run_metadata.json")

print("Reports zip:", reports_zip)
print("Model zip  :", model_zip)
print()
print("Download from the Colab file browser (folder icon, left panel):")
print("  /content/drive/MyDrive/" + BASE_DIR.name + "/")

# Optional: trigger a browser download instead of using the file browser.
DOWNLOAD_IN_BROWSER = False
if DOWNLOAD_IN_BROWSER:
    from google.colab import files

    files.download(reports_zip)
    files.download(model_zip)

## Step 13 — Bring the results home and verify

Download `reports_<run>.zip` and `model_<run>.zip` from
`Drive/...-outputs/`, then on your machine:

```powershell
# restore the Git-trackable reports
Expand-Archive .\reports_baseline_<stamp>.zip `
  -DestinationPath .\models\metadata\runs\baseline_<stamp>

# keep the weights next to the other ignored binaries (stays out of Git)
Expand-Archive .\model_baseline_<stamp>.zip -DestinationPath .\models
```

**Pending verification — run before ever claiming "the model loads" locally:**

```powershell
.venv\Scripts\python.exe -c "import tensorflow as tf; m = tf.keras.models.load_model(r'models\best_model.keras'); print(m.output_shape)"
```

### Reproducibility limits (read before comparing numbers)

- Each run records its own environment (`environment_freeze.txt` and
  `run_metadata.json` → `environment`). A Colab runtime is **not** the local
  Windows/Python 3.11 environment; different GPUs and library builds change
  floating-point results slightly even with seed 42.
- Early stopping makes the epoch count depend on data + hardware — expect
  different stopping epochs across environments.
- `REVISION = "main"` is unpinned unless you set a commit SHA.

### Honesty statements

- This notebook has **not been executed in Google Colab**. It was validated
  locally: valid JSON, every code cell passes a Python syntax check, and
  `tests/test_notebook.py` passes. It deliberately contains **no accuracy
  numbers**.
- The baseline reports **validation** metrics only; the test split is
  evaluated in a later milestone.
- Any accuracy figure you later read here must come from that run's own
  `run_metadata.json` — never from a different run or another project.